# Netflix Content Type Prediction (Movie vs TV Show)
## End-to-End Supervised Machine Learning Pipeline

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

ModuleNotFoundError: No module named 'pandas'

### Phase 1 & 2: Load Data & Apply Target Leakage Prevention Protocol

In [ ]:
df_raw = pd.read_csv("netflix_titles.csv")
print("Raw Dataset Shape:", df_raw.shape)

# Filter valid target values and map target: Movie -> 0, TV Show -> 1
df = df_raw[df_raw['type'].isin(['Movie', 'TV Show'])].copy()
df['target'] = df['type'].map({'Movie': 0, 'TV Show': 1})

# Target Leakage Removal Protocol:
# Drop 'duration' (explicit leakage: min vs Seasons)
# Drop 'show_id', 'title', 'description' (non-predictive or raw text)
leakage_cols = ['duration', 'show_id', 'title', 'description', 'type']
df_clean = df.drop(columns=leakage_cols)
print("Cleaned Columns:", df_clean.columns.tolist())

### Phase 3: Feature Engineering & Preprocessing Pipeline

In [ ]:
# Extract primary country
df_clean['primary_country'] = df_clean['country'].fillna('Unknown').apply(
    lambda x: x.split(',')[0].strip() if isinstance(x, str) else 'Unknown'
)

# Extract primary genre
df_clean['primary_genre'] = df_clean['listed_in'].fillna('Unknown').apply(
    lambda x: x.split(',')[0].strip() if isinstance(x, str) else 'Unknown'
)

# Parse date_added for year and month
df_clean['date_added_parsed'] = pd.to_datetime(df_clean['date_added'].str.strip(), errors='coerce')
df_clean['year_added'] = df_clean['date_added_parsed'].dt.year
df_clean['month_added'] = df_clean['date_added_parsed'].dt.month_name().fillna('Unknown')

# Binary flag for director
df_clean['has_director'] = df_clean['director'].notnull().astype(int)

num_features = ['release_year', 'year_added', 'has_director']
cat_features = ['rating', 'primary_country', 'primary_genre', 'month_added']

X = df_clean[num_features + cat_features]
y = df_clean['target']

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

### Phase 4 & 5: Model Training, Pipelines & Performance Evaluation

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_split=10, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_split=5, class_weight='balanced', random_state=42, n_jobs=-1)
}

results = []
trained_pipelines = {}

for name, clf in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})
    trained_pipelines[name] = (pipeline, f1)

results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
print("\n--- Model Performance Metrics ---")
print(results_df.to_string(index=False))

### Phase 6: Save Model Artifact

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pipeline, best_f1 = trained_pipelines[best_model_name]

os.makedirs("saved_models", exist_ok=True)
model_path = os.path.join("saved_models", "netflix_content_type_model.joblib")
joblib.dump(best_pipeline, model_path)
print(f"Best Model ('{best_model_name}') saved to '{model_path}' successfully!")